In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

df = pd.read_parquet("../data/processed/cleaned_flows.parquet")
print(df.shape)
print(df.dtypes.value_counts())

(2520724, 80)
int32      27
int8       18
float64    15
float32     9
int16       7
int64       2
str         2
Name: count, dtype: int64


In [2]:
# Everything except the two label columns is a candidate feature
feature_cols = [c for c in df.columns if c not in ['Label', 'Label_binary']]

# Check if any features are still non-numeric (would need encoding)
non_numeric = df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric feature columns:", non_numeric)

Non-numeric feature columns: []


In [3]:
X = df[feature_cols]
y = df['Label_binary']

# 70/15/15 split, stratified to preserve the ATTACK/BENIGN ratio in every split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"\nTrain distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nVal distribution:\n{y_val.value_counts(normalize=True)}")
print(f"\nTest distribution:\n{y_test.value_counts(normalize=True)}")

Train: (1764506, 78), Val: (378109, 78), Test: (378109, 78)

Train distribution:
Label_binary
BENIGN    0.831131
ATTACK    0.168869
Name: proportion, dtype: float64

Val distribution:
Label_binary
BENIGN    0.831131
ATTACK    0.168869
Name: proportion, dtype: float64

Test distribution:
Label_binary
BENIGN    0.831131
ATTACK    0.168869
Name: proportion, dtype: float64


In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"Train mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"Train std (should be ~1): {X_train_scaled.std():.4f}")

Scaling complete.
Train mean (should be ~0): 0.0000
Train std (should be ~1): 0.9473


In [5]:
import joblib

joblib.dump(scaler, "../data/processed/scaler_binary.joblib")

np.save("../data/processed/X_train_binary.npy", X_train_scaled)
np.save("../data/processed/X_val_binary.npy", X_val_scaled)
np.save("../data/processed/X_test_binary.npy", X_test_scaled)
y_train.to_csv("../data/processed/y_train_binary.csv", index=False)
y_val.to_csv("../data/processed/y_val_binary.csv", index=False)
y_test.to_csv("../data/processed/y_test_binary.csv", index=False)

print("Saved all binary preprocessing artifacts.")

Saved all binary preprocessing artifacts.
